In [1]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, Embedding, Conv1D, MaxPooling1D, Flatten, Dense, Dropout

In [2]:
import tensorflow as tf

In [3]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [4]:
from keras.layers import Conv2D
from keras.layers import MaxPooling2D

In [5]:
cnn= Sequential()

In [6]:
cnn.add(Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[128, 128, 3]))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

g:\My Drive\Bone_fracture\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
cnn.add(Conv2D(filters=64, kernel_size=3, activation='relu'))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

In [8]:
cnn.add(Conv2D(filters=128, kernel_size=3, activation='relu'))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

In [9]:
cnn.add(Flatten())

In [10]:
cnn.add(Dense(256,activation='relu'))
cnn.add(Dropout(0.5))

cnn.add(Dense(128,activation='relu'))
cnn.add(Dense(1,activation ='sigmoid'))

cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [11]:
cnn.optimizer

In [12]:
batch_size = 16

# this is the augmentation configuration we will use for training
train_datagen = ImageDataGenerator(
        rescale=1./255,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True)

# this is the augmentation configuration we will use for testing:
# only rescaling
test_datagen = ImageDataGenerator(rescale=1./255)

# this is a generator that will read pictures found in
# subfolers of 'data/train', and indefinitely generate
# batches of augmented image data
train_generator = train_datagen.flow_from_directory(
        r'G:\My Drive\Bone_fracture\train',  # this is the target directory
        target_size=(128, 128),  # all images will be resized to 128x128
        batch_size=batch_size,
        class_mode='binary')  # since we use binary_crossentropy loss, we need binary labels

# this is a similar generator, for validation data
validation_generator = test_datagen.flow_from_directory(
        r'G:\My Drive\Bone_fracture\val',
        target_size=(128, 128),
        batch_size=batch_size,
        class_mode='binary')

Found 8863 images belonging to 2 classes.
Found 600 images belonging to 2 classes.


In [13]:
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,549,057 (24.98 MB)

 Trainable params: 6,549,057 (24.98 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
early_stopping = EarlyStopping(
    monitor='val_loss',          # Monitor validation loss
    patience=3,                  # Wait for 3 epochs before stopping
    restore_best_weights=True,   # Restore the best model weights
    verbose=1
)

In [19]:
cnn.fit(
        train_generator,
        epochs=10,
        validation_data=validation_generator,
        callbacks=[early_stopping]
        )

Epoch 1/10
554/554 ━━━━━━━━━━━━━━━━━━━━ 280s 504ms/step - accuracy: 0.9859 - loss: 0.0429 - val_accuracy: 0.6083 - val_loss: 1.8731
Epoch 2/10
554/554 ━━━━━━━━━━━━━━━━━━━━ 292s 528ms/step - accuracy: 0.9853 - loss: 0.0480 - val_accuracy: 0.6433 - val_loss: 1.4438
Epoch 3/10
554/554 ━━━━━━━━━━━━━━━━━━━━ 298s 538ms/step - accuracy: 0.9877 - loss: 0.0345 - val_accuracy: 0.6867 - val_loss: 1.2224
Epoch 4/10
554/554 ━━━━━━━━━━━━━━━━━━━━ 299s 541ms/step - accuracy: 0.9857 - loss: 0.0394 - val_accuracy: 0.6283 - val_loss: 1.6027
Epoch 5/10
554/554 ━━━━━━━━━━━━━━━━━━━━ 299s 539ms/step - accuracy: 0.9859 - loss: 0.0383 - val_accuracy: 0.5933 - val_loss: 2.0907
Epoch 6/10
554/554 ━━━━━━━━━━━━━━━━━━━━ 312s 564ms/step - accuracy: 0.9892 - loss: 0.0305 - val_accuracy: 0.6733 - val_loss: 1.6110
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 3.


In [15]:
from keras.preprocessing import image
import numpy as np

In [20]:
img = image.load_img(r'G:\My Drive\Bone_fracture\broken-bones-s2-fractures.webp', target_size=(128, 128,3))
img= image.img_to_array(img)
img = img/255
img = np.expand_dims(img, axis=0)
prediction = cnn.predict(img)
if prediction[0][0] > 0.5:
    print("The bone is not broken")
else:
    print("The bone is broken")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 653ms/step
The bone is broken


In [23]:
print(train_generator)
print(validation_generator)

In [24]:
print(train_generator.samples)
print(validation_generator.samples)

8863
600


In [25]:
print(train_generator.class_indices)

{'fractured': 0, 'not fractured': 1}


In [ ]:
print(tf.__version__)

2.19.0


In [28]:
print(train_generator.samples)
print(validation_generator.samples)

8863
600
